# Get Edmonton Restautants Data from Google Map

In [1]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
API_KEY = os.getenv("API_KEY")

In [3]:
search_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"

params = {
    "radius": 3500,
    "type": "restaurant",
    "key": API_KEY
}

grid_points = [
    (53.60, -113.60), (53.60, -113.55), (53.60, -113.50), (53.60, -113.45), (53.60, -113.40),
    (53.55, -113.60), (53.55, -113.55), (53.55, -113.50), (53.55, -113.45), (53.55, -113.40),
    (53.50, -113.60), (53.50, -113.55), (53.50, -113.50), (53.50, -113.45), (53.50, -113.40),
    (53.45, -113.60), (53.45, -113.55), (53.45, -113.50), (53.45, -113.45), (53.45, -113.40)
]

In [4]:
restaurants = []

for lat, lng in grid_points:
    params["location"] = f"{lat},{lng}"
    while True:
        response = requests.get(search_url, params=params).json()
        restaurants += response['results']

        if 'next_page_token' in response.keys():
            time.sleep(2)
            params["pagetoken"] = response["next_page_token"]
        else:
            break
    
    if 'pagetoken' in params:
        del params['pagetoken']

In [5]:
restaurants_id = [restaurant['place_id'] for restaurant in restaurants]
restaurants_id = set(restaurants_id)

In [6]:
restaurants_details = []
for place_id in restaurants_id:
    details_url = f"https://places.googleapis.com/v1/places/{place_id}"
    
    params = {
        "key": API_KEY,
        "fields": "id,displayName,formattedAddress,types,accessibilityOptions,primaryType,primaryTypeDisplayName,priceLevel,priceRange,rating,regularOpeningHours,userRatingCount,allowsDogs,curbsidePickup,delivery,dineIn,editorialSummary,goodForChildren,goodForGroups,goodForWatchingSports,liveMusic,menuForChildren,parkingOptions,paymentOptions,outdoorSeating,reservable,restroom,servesBeer,servesBreakfast,servesBrunch,servesCocktails,servesCoffee,servesDessert,servesDinner,servesLunch,servesVegetarianFood,servesWine,takeout"
    }

    response = requests.get(details_url, params=params).json()
    
    restaurants_details.append(response)

In [7]:
df = pd.DataFrame(restaurants_details)

In [11]:
df.head()

,id,types,formattedAddress,rating,regularOpeningHours,priceLevel,userRatingCount,displayName,primaryTypeDisplayName,takeout,...,allowsDogs,restroom,goodForGroups,goodForWatchingSports,paymentOptions,parkingOptions,accessibilityOptions,priceRange,servesVegetarianFood,editorialSummary
0,ChIJQWNpDe0YoFMRto7xgtIAHEk,"[breakfast_restaurant, restaurant, point_of_in...","5706 75 Street NW, Edmonton, AB T6E 5X6, Canada",4.0,"{'openNow': True, 'periods': [{'open': {'day':...",PRICE_LEVEL_INEXPENSIVE,749.0,{'text': 'Ecity Flame (Best Offer on Website)'...,"{'text': 'Restaurant', 'languageCode': 'en-US'}",True,...,False,True,True,True,"{'acceptsCreditCards': True, 'acceptsDebitCard...","{'freeParkingLot': True, 'paidParkingLot': Fal...","{'wheelchairAccessibleParking': True, 'wheelch...","{'startPrice': {'currencyCode': 'CAD', 'units'...",NaN,NaN
1,ChIJQ2rvH4cYoFMRsV6eIIgjmRs,"[gas_station, convenience_store, fast_food_res...","7535 75 Street NW, Edmonton, AB T6C 4M3, Canada",3.4,"{'openNow': True, 'periods': [{'open': {'day':...",PRICE_LEVEL_MODERATE,95.0,"{'text': 'Shell', 'languageCode': 'en'}","{'text': 'Gas Station', 'languageCode': 'en-US'}",True,...,NaN,True,NaN,NaN,"{'acceptsCreditCards': True, 'acceptsDebitCard...","{'freeParkingLot': True, 'freeStreetParking': ...",{'wheelchairAccessibleRestroom': True},NaN,NaN,NaN
2,ChIJt4Bu_ksZoFMRR0pb8OvRXoQ,"[bar_and_grill, bar, restaurant, point_of_inte...","1107 Knottwood Rd E Northwest, Edmonton, AB T6...",4.3,"{'openNow': True, 'periods': [{'open': {'day':...",PRICE_LEVEL_MODERATE,884.0,"{'text': 'JT's Bar & Grill', 'languageCode': '...","{'text': 'Bar & Grill', 'languageCode': 'en-US'}",True,...,False,True,True,True,"{'acceptsCreditCards': True, 'acceptsDebitCard...","{'freeParkingLot': True, 'freeStreetParking': ...","{'wheelchairAccessibleParking': True, 'wheelch...","{'startPrice': {'currencyCode': 'CAD', 'units'...",True,NaN
3,ChIJD_onNiskoFMRz9s6W7Si_Qw,"[pizza_restaurant, fast_food_restaurant, meal_...","2 Hebert Rd, St. Albert, AB T8N 6T7, Canada",3.3,"{'openNow': True, 'periods': [{'open': {'day':...",PRICE_LEVEL_INEXPENSIVE,84.0,"{'text': 'Little Caesars Pizza', 'languageCode...","{'text': 'Pizza Restaurant', 'languageCode': '...",True,...,False,NaN,NaN,False,"{'acceptsCreditCards': True, 'acceptsDebitCard...","{'freeParkingLot': True, 'freeStreetParking': ...","{'wheelchairAccessibleParking': True, 'wheelch...","{'startPrice': {'currencyCode': 'CAD', 'units'...",NaN,{'text': 'Carry-out chain featuring chicken wi...
4,ChIJiXfDGDQhoFMRehR_Gj75DHs,"[convenience_store, hotel, food_store, lodging...","16806 118 Ave NW, Edmonton, AB T5V 1M8, Canada",3.3,"{'openNow': True, 'periods': [{'open': {'day':...",NaN,15.0,"{'text': 'West Edmonton Truckland Ltd', 'langu...","{'text': 'Convenience Store', 'languageCode': ...",True,...,NaN,True,NaN,NaN,"{'acceptsCreditCards': True, 'acceptsDebitCard...","{'freeParkingLot': True, 'freeStreetParking': ...","{'wheelchairAccessibleParking': True, 'wheelch...",NaN,NaN,NaN


In [12]:
df.to_json('data/raw_yeg_restaurants.json')